In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
import sys
# import time

project_path = os.path.join(os.getcwd(), '..', '..')
sys.path.append(project_path)

from utils.transformations import reusable

In [0]:
chkptloc_user = "abfss://silver@atharproject.dfs.core.windows.net/DimUser/checkpoint"
chkptloc_artist = "abfss://silver@atharproject.dfs.core.windows.net/DimArtist/checkpoint"
chkptloc_track = "abfss://silver@atharproject.dfs.core.windows.net/DimTrack/checkpoint"
chkptloc_date = "abfss://silver@atharproject.dfs.core.windows.net/DimDate/checkpoint"
chkptloc_fact = "abfss://silver@atharproject.dfs.core.windows.net/FactStream/checkpoint"

###DimUser

In [0]:
df_user = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format", "parquet")\
            .option("cloudFiles.schemaLocation", "abfss://silver@atharproject.dfs.core.windows.net/DimUser/schema")\
            .option("schemaEvolutionMode", "rescue")\
            .load("abfss://bronze@atharproject.dfs.core.windows.net/DimUser")

In [0]:
df_user_obj = reusable()
df_user = df_user_obj.dropColumns(df_user, ["_rescued_data"])
df_user = df_user.dropDuplicates(["user_id"])

In [0]:
df_user.writeStream.format("delta")\
      .outputMode("append")\
      .option("checkpointLocation", chkptloc_user)\
      .option("path", "abfss://silver@atharproject.dfs.core.windows.net/DimUser/data")\
      .option("mergeSchema", "true")\
      .trigger(once=True)\
      .toTable("spotify_cata.silver.dim_user")

###DimArtist

In [0]:
df_artist = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format", "parquet")\
            .option("cloudFiles.schemaLocation", "abfss://silver@atharproject.dfs.core.windows.net/DimArtist/schema")\
            .option("schemaEvolutionMode", "rescue")\
            .load("abfss://bronze@atharproject.dfs.core.windows.net/DimArtist")

In [0]:
df_artist_obj = reusable()
df_artist = df_artist_obj.dropColumns(df_artist, ["_rescued_data"])
df_artist = df_artist.dropDuplicates(["artist_id"])

In [0]:
df_artist.writeStream.format("delta")\
      .outputMode("append")\
      .option("checkpointLocation", chkptloc_artist)\
      .option("path", "abfss://silver@atharproject.dfs.core.windows.net/DimArtist/data")\
      .option("mergeSchema", "true")\
      .trigger(once=True)\
      .toTable("spotify_cata.silver.dim_artist")

###DimTrack

In [0]:
df_track = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format", "parquet")\
            .option("cloudFiles.schemaLocation", "abfss://silver@atharproject.dfs.core.windows.net/DimTrack/schema")\
            .option("schemaEvolutionMode", "rescue")\
            .load("abfss://bronze@atharproject.dfs.core.windows.net/DimTrack")

In [0]:
df_track = df_track.withColumn("durationFlag", when(col("duration_sec") < 150, "short")\
                                            .when(col("duration_sec") < 300, "medium")\
                                            .otherwise("long")         
)

df_track = df_track.withColumn("track_name", regexp_replace(col("track_name"), "-", " "))

In [0]:
df_track_obj = reusable()
df_track = df_track_obj.dropColumns(df_track, ["_rescued_data"])
df_track = df_track.dropDuplicates(["track_id"])

In [0]:
df_track.writeStream.format("delta")\
      .outputMode("append")\
      .option("checkpointLocation", chkptloc_track)\
      .option("mergeSchema", "true")\
      .option("path", "abfss://silver@atharproject.dfs.core.windows.net/DimTrack/data")\
      .trigger(once=True)\
      .toTable("spotify_cata.silver.dim_track")

###DimDate

In [0]:
df_date = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format", "parquet")\
            .option("cloudFiles.schemaLocation", "abfss://silver@atharproject.dfs.core.windows.net/DimDate/schema")\
            .option("schemaEvolutionMode", "rescue")\
            .load("abfss://bronze@atharproject.dfs.core.windows.net/DimDate")

In [0]:
df_date_obj = reusable()
df_date = df_date_obj.dropColumns(df_date, ["_rescued_data"])

In [0]:
df_date.writeStream.format("delta")\
      .outputMode("append")\
      .option("checkpointLocation", chkptloc_date)\
      .option("path", "abfss://silver@atharproject.dfs.core.windows.net/DimDate/data")\
      .option("mergeSchema", "true")\
      .trigger(once=True)\
      .toTable("spotify_cata.silver.dim_date")

###FactStream

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format", "parquet")\
            .option("cloudFiles.schemaLocation", "abfss://silver@atharproject.dfs.core.windows.net/FactStream/schema")\
            .option("schemaEvolutionMode", "rescue")\
            .load("abfss://bronze@atharproject.dfs.core.windows.net/FactStream")

In [0]:
df_fact_obj = reusable()
df_fact = df_fact_obj.dropColumns(df_fact, ["_rescued_data"])

In [0]:
df_fact.writeStream.format("delta")\
      .outputMode("append")\
      .option("checkpointLocation", chkptloc_fact)\
      .option("path", "abfss://silver@atharproject.dfs.core.windows.net/FactStream/data")\
      .option("mergeSchema", "true")\
      .trigger(once=True)\
      .toTable("spotify_cata.silver.fact_stream")